# Laboratorio Neo4j

Il laboratorio presenta le caratteristiche fondamentali di **Neo4j**, il database a grafo più diffuso, usando il linguaggio di interrogazione **Cypher** e il driver ufficiale Python.

**Indice**

1. Il modello a grafo di proprietà
2. Connessione al database
3. Operazioni CRUD: `CREATE`, `MATCH`, `SET`, `REMOVE`, `DELETE`, `MERGE`
4. Parametri e transazioni
5. Schema: vincoli e indici
6. Importazione di dati da CSV (`LOAD CSV`)
7. Interrogazioni: filtri, aggregazioni, sottoquery
8. Navigare il grafo: cammini e cammini minimi
9. Visualizzare il grafo
10. Neo4j Browser

> Le celle vanno eseguite **in ordine**, dall'alto verso il basso. In caso di problemi è possibile ripartire da capo con *Run → Run All Cells*: il notebook svuota il database all'inizio.

## 1. Il modello a grafo di proprietà

Neo4j memorizza i dati come un **grafo di proprietà** (*property graph*):

| Elemento | Descrizione | Esempio |
|---|---|---|
| **Nodo** | un'entità | uno studente, un corso |
| **Etichetta** (*label*) | il "tipo" di un nodo; un nodo può averne più di una | `:Studente`, `:Persona` |
| **Relazione** | un collegamento **orientato** e **tipizzato** tra due nodi | `(:Studente)-[:ISCRITTO_A]->(:Corso)` |
| **Proprietà** | coppie chiave/valore su nodi *e* relazioni | `nome: "Elena"`, `voto: 28` |

A differenza di un database relazionale non esistono tabelle né JOIN: le relazioni sono memorizzate fisicamente come puntatori tra nodi (*index-free adjacency*), quindi attraversarle costa poco anche su grafi molto grandi.

In Cypher i pattern si scrivono in "ASCII art": i nodi tra parentesi tonde `( )`, le relazioni tra parentesi quadre `-[ ]->`:

```cypher
(s:Studente {nome: "Elena"})-[:ISCRITTO_A {voto: 28}]->(c:Corso {titolo: "Basi di Dati"})
```

## 2. Connessione al database

Neo4j espone due porte:

* **7687 – Bolt**: il protocollo binario usato dai driver (Python, Java, JavaScript, ...);
* **7474 – HTTP**: l'interfaccia web *Neo4j Browser* (vedi sezione 10).

Il notebook gira in un container separato: per raggiungere Neo4j si usa il nome del servizio Docker (`neo4j`) invece di `localhost`. I parametri di connessione arrivano dalle variabili d'ambiente definite in `docker-compose.yml`.

In [ ]:
import os
import neo4j
from neo4j import GraphDatabase

URI = os.environ.get("NEO4J_URI", "neo4j://localhost:7687")
AUTH = (os.environ.get("NEO4J_USER", "neo4j"), os.environ.get("NEO4J_PASSWORD", "laboratorio"))

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()

info = driver.get_server_info()
print(f"Connesso a {info.address} - {info.agent} (driver Python {neo4j.__version__})")

Il modo più semplice per eseguire una query è `driver.execute_query()`. Restituisce i **record**, un **riassunto** (*summary*) con statistiche e tempi, e le **chiavi** (i nomi delle colonne).

In [ ]:
records, summary, keys = driver.execute_query("RETURN 'Ciao, Neo4j!' AS saluto, 1 + 2 AS somma")

print(keys)
print(records[0]["saluto"], records[0]["somma"])
print(f"Eseguita in {summary.result_available_after} ms")

Per comodità viene definita una funzione `run()` che restituisce il risultato come **DataFrame pandas**, molto più leggibile in un notebook.

In [ ]:
import pandas as pd

def run(query, **params):
    """Esegue una query Cypher e restituisce il risultato come DataFrame."""
    return driver.execute_query(query, params, result_transformer_=neo4j.Result.to_df)

run("CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition")

Si parte da un database vuoto. `DETACH DELETE` elimina i nodi **insieme** alle loro relazioni; vengono eliminati anche vincoli e indici creati in esecuzioni precedenti.

In [ ]:
driver.execute_query("MATCH (n) DETACH DELETE n")
for c in run("SHOW CONSTRAINTS YIELD name")["name"]:
    run(f"DROP CONSTRAINT `{c}`")
for i in run("SHOW INDEXES YIELD name, type WHERE type <> 'LOOKUP'")["name"]:
    run(f"DROP INDEX `{i}`")

run("MATCH (n) RETURN count(n) AS nodi")

## 3. Operazioni CRUD

### CREATE: creare nodi e relazioni

In [ ]:
summary = driver.execute_query("""
    CREATE (a:Persona {nome: 'Alice', eta: 23})
    CREATE (b:Persona {nome: 'Bruno', eta: 25})
    CREATE (c:Persona:Docente {nome: 'Carla', eta: 41})   // due etichette
    CREATE (a)-[:CONOSCE {dal: 2022}]->(b)
    CREATE (b)-[:CONOSCE {dal: 2023}]->(c)
""").summary

print(summary.counters)

### MATCH: leggere

`MATCH` cerca nel grafo tutte le occorrenze di un pattern; `WHERE` filtra; `RETURN` sceglie cosa restituire.

In [ ]:
run("""
    MATCH (p:Persona)
    WHERE p.eta < 30
    RETURN p.nome AS nome, p.eta AS eta, labels(p) AS etichette
    ORDER BY eta
""")

In [ ]:
# Un pattern con una relazione: chi conosce chi, e da quando?
run("""
    MATCH (a:Persona)-[r:CONOSCE]->(b:Persona)
    RETURN a.nome AS chi, type(r) AS relazione, b.nome AS chi_conosce, r.dal AS dal
""")

### SET e REMOVE: modificare proprietà ed etichette

In [ ]:
run("""
    MATCH (a:Persona {nome: 'Alice'})
    SET a.email = 'alice@example.org', a.eta = a.eta + 1, a:Studente
    REMOVE a.eta
    RETURN a.nome AS nome, a.email AS email, a.eta AS eta, labels(a) AS etichette
""")

Nota: Neo4j è *schema-optional*. Due nodi con la stessa etichetta possono avere proprietà diverse e una proprietà mancante vale semplicemente `null`.

### MERGE: creare solo se non esiste

`MERGE` è un "trova oppure crea": è **idempotente**, quindi eseguirlo più volte non produce duplicati. `ON CREATE` e `ON MATCH` permettono di comportarsi in modo diverso nei due casi.

In [ ]:
query = """
    MERGE (p:Persona {nome: 'Davide'})
    ON CREATE SET p.creato = datetime(), p.visite = 1
    ON MATCH  SET p.visite = p.visite + 1
    RETURN p.nome AS nome, p.visite AS visite
"""
for _ in range(3):
    print(run(query).to_dict("records"))

run("MATCH (p:Persona {nome: 'Davide'}) RETURN count(p) AS quanti_davide")

### DELETE e DETACH DELETE

Un nodo con relazioni non può essere eliminato con un semplice `DELETE` (lascerebbe relazioni "appese"): Neo4j solleva un errore. Serve `DETACH DELETE`.

In [ ]:
try:
    run("MATCH (b:Persona {nome: 'Bruno'}) DELETE b")
except neo4j.exceptions.Neo4jError as e:
    print("Errore atteso:", e.code)

run("MATCH (b:Persona {nome: 'Bruno'}) DETACH DELETE b")
run("MATCH (p:Persona) RETURN p.nome AS rimasti")

In [ ]:
# Pulizia prima di passare al dataset vero e proprio
driver.execute_query("MATCH (n) DETACH DELETE n").summary.counters

## 4. Parametri e transazioni

### Parametri

I valori non vanno **mai** concatenati nel testo della query: si usano i **parametri** (`$nome`). Sono più sicuri (niente *Cypher injection*) e più veloci, perché Neo4j riusa il piano di esecuzione già calcolato.

In [ ]:
persone = [{"nome": "Alice", "eta": 23}, {"nome": "Bruno", "eta": 25}, {"nome": "Carla", "eta": 41}]

# UNWIND trasforma una lista in righe: un modo efficiente per inserire molti dati in una sola query
driver.execute_query("""
    UNWIND $persone AS p
    CREATE (:Persona {nome: p.nome, eta: p.eta})
""", persone=persone)

run("MATCH (p:Persona) WHERE p.eta >= $min RETURN p.nome AS nome", min=24)

### Transazioni

Ogni query in Neo4j gira in una transazione **ACID**. Con una *session* è possibile raggruppare più istruzioni in un'unica transazione: o vanno a buon fine tutte, o nessuna.

`execute_write` / `execute_read` gestiscono la transazione automaticamente (commit automatico e nuovi tentativi in caso di errori temporanei).

In [ ]:
def trasferisci_amicizia(tx, da, a):
    tx.run("MATCH (x:Persona {nome: $da}), (y:Persona {nome: $a}) MERGE (x)-[:CONOSCE]->(y)", da=da, a=a)
    return tx.run("MATCH (:Persona {nome: $da})-[:CONOSCE]->(y) RETURN collect(y.nome) AS amici", da=da).single()["amici"]

with driver.session() as session:
    print(session.execute_write(trasferisci_amicizia, "Alice", "Bruno"))

Se una funzione di transazione solleva un'eccezione, **tutte** le sue modifiche vengono annullate (*rollback*):

In [ ]:
def operazione_fallita(tx):
    tx.run("CREATE (:Persona {nome: 'Fantasma'})")
    raise ValueError("qualcosa è andato storto a metà transazione")

with driver.session() as session:
    try:
        session.execute_write(operazione_fallita)
    except ValueError as e:
        print("Eccezione:", e)

# Il nodo 'Fantasma' non esiste: la transazione è stata annullata
run("MATCH (p:Persona {nome: 'Fantasma'}) RETURN count(p) AS fantasmi")

In [ ]:
driver.execute_query("MATCH (n) DETACH DELETE n").summary.counters

## 5. Schema: vincoli e indici

Anche se Neo4j non impone uno schema, conviene definire:

* **vincoli di unicità**: garantiscono che una proprietà identifichi un nodo (come una chiave primaria) e creano automaticamente un indice;
* **indici**: velocizzano la ricerca dei nodi da cui far partire una query.

Conviene crearli **prima** di importare i dati, così l'import stesso è più veloce.

In [ ]:
schema = [
    "CREATE CONSTRAINT studente_matricola IF NOT EXISTS FOR (s:Studente) REQUIRE s.matricola IS UNIQUE",
    "CREATE CONSTRAINT corso_codice IF NOT EXISTS FOR (c:Corso) REQUIRE c.codice IS UNIQUE",
    "CREATE CONSTRAINT docente_id IF NOT EXISTS FOR (d:Docente) REQUIRE d.id IS UNIQUE",
    "CREATE CONSTRAINT dipartimento_codice IF NOT EXISTS FOR (d:Dipartimento) REQUIRE d.codice IS UNIQUE",
    "CREATE INDEX corso_titolo IF NOT EXISTS FOR (c:Corso) ON (c.titolo)",
]
for q in schema:
    driver.execute_query(q)

# Gli indici di tipo LOOKUP sono creati automaticamente da Neo4j per etichette e tipi di relazione
run("SHOW INDEXES YIELD name, type, labelsOrTypes, properties, owningConstraint WHERE type <> 'LOOKUP'")

## 6. Importare dati da CSV

Il laboratorio usa un piccolo dataset di esempio su un ateneo **fittizio**, contenuto nella cartella `import/` del repository (montata nel container Neo4j in `/import`):

| File | Contenuto |
|---|---|
| `dipartimenti.csv` | i dipartimenti |
| `docenti.csv` | i docenti e il loro dipartimento |
| `corsi.csv` | i corsi, i CFU e il docente titolare |
| `studenti.csv` | gli studenti e il dipartimento di afferenza |
| `iscrizioni.csv` | quali corsi segue ogni studente ed eventuale voto |
| `amicizie.csv` | le amicizie tra studenti |

Il modello da ottenere:

```
(:Studente)-[:AMICO_DI {dal}]->(:Studente)
(:Studente)-[:ISCRITTO_A {voto}]->(:Corso)
(:Studente)-[:AFFERISCE_A]->(:Dipartimento)
(:Docente)-[:INSEGNA]->(:Corso)
(:Docente)-[:AFFERISCE_A]->(:Dipartimento)
```

`LOAD CSV WITH HEADERS` legge il file riga per riga: ogni riga è una mappa con chiavi uguali all'intestazione. **Tutti i valori sono stringhe**: vanno convertiti con `toInteger()`, `toFloat()`, `date()`, ... Le celle vuote diventano `null`.

In [ ]:
import_nodi = [
    """LOAD CSV WITH HEADERS FROM 'file:///dipartimenti.csv' AS row
       MERGE (d:Dipartimento {codice: row.codice}) SET d.nome = row.nome""",

    """LOAD CSV WITH HEADERS FROM 'file:///docenti.csv' AS row
       MATCH (dip:Dipartimento {codice: row.dipartimento})
       MERGE (d:Docente {id: row.id}) SET d.nome = row.nome
       MERGE (d)-[:AFFERISCE_A]->(dip)""",

    """LOAD CSV WITH HEADERS FROM 'file:///corsi.csv' AS row
       MATCH (doc:Docente {id: row.docente})
       MERGE (c:Corso {codice: row.codice}) SET c.titolo = row.titolo, c.cfu = toInteger(row.cfu)
       MERGE (doc)-[:INSEGNA]->(c)""",

    """LOAD CSV WITH HEADERS FROM 'file:///studenti.csv' AS row
       MATCH (dip:Dipartimento {codice: row.dipartimento})
       MERGE (s:Studente {matricola: row.matricola})
       SET s.nome = row.nome, s.anno_nascita = toInteger(row.anno_nascita)
       MERGE (s)-[:AFFERISCE_A]->(dip)""",
]
for q in import_nodi:
    print(driver.execute_query(q).summary.counters)

Per file grandi conviene spezzare l'import in tante piccole transazioni con `CALL { ... } IN TRANSACTIONS`. Questa clausola richiede una transazione **implicita** (*auto-commit*), che si ottiene con `session.run()`.

In [ ]:
import_relazioni = [
    """LOAD CSV WITH HEADERS FROM 'file:///iscrizioni.csv' AS row
       CALL (row) {
           MATCH (s:Studente {matricola: row.matricola}), (c:Corso {codice: row.corso})
           MERGE (s)-[r:ISCRITTO_A]->(c)
           SET r.voto = toInteger(row.voto)
       } IN TRANSACTIONS OF 50 ROWS""",

    """LOAD CSV WITH HEADERS FROM 'file:///amicizie.csv' AS row
       CALL (row) {
           MATCH (a:Studente {matricola: row.da}), (b:Studente {matricola: row.a})
           MERGE (a)-[r:AMICO_DI]->(b)
           SET r.dal = toInteger(row.dal)
       } IN TRANSACTIONS OF 50 ROWS""",
]
with driver.session() as session:
    for q in import_relazioni:
        print(session.run(q).consume().counters)

In [ ]:
# Quanti nodi e relazioni per tipo?
display(run("MATCH (n) RETURN labels(n)[0] AS etichetta, count(*) AS nodi ORDER BY nodi DESC"))
display(run("MATCH ()-[r]->() RETURN type(r) AS tipo, count(*) AS relazioni ORDER BY relazioni DESC"))

Neo4j sa descrivere il proprio schema, ricavandolo dai dati:

In [ ]:
run("""
    CALL db.schema.visualization() YIELD relationships
    UNWIND relationships AS r
    RETURN DISTINCT labels(startNode(r))[0] AS da, type(r) AS relazione, labels(endNode(r))[0] AS verso
""")

## 7. Interrogazioni

### Filtri, ordinamento, paginazione

In [ ]:
run("""
    MATCH (s:Studente)-[:AFFERISCE_A]->(:Dipartimento {codice: 'INF'})
    WHERE s.anno_nascita >= 2002 OR s.nome CONTAINS 'Ricci'
    RETURN s.matricola AS matricola, s.nome AS nome, s.anno_nascita AS anno
    ORDER BY anno DESC, nome
    LIMIT 10
""")

### Aggregazioni

Non esiste `GROUP BY`: Cypher raggruppa automaticamente per tutte le espressioni del `RETURN` che **non** sono funzioni di aggregazione (`count`, `avg`, `sum`, `min`, `max`, `collect`, ...).

In [ ]:
# Media dei voti e numero di iscritti per corso
run("""
    MATCH (s:Studente)-[r:ISCRITTO_A]->(c:Corso)
    RETURN c.titolo AS corso,
           count(s) AS iscritti,
           count(r.voto) AS esami_superati,      // count(espressione) ignora i null
           round(avg(r.voto), 1) AS media_voti
    ORDER BY media_voti DESC
""")

In [ ]:
# collect() costruisce liste: i corsi di ogni docente
run("""
    MATCH (d:Docente)-[:INSEGNA]->(c:Corso)
    RETURN d.nome AS docente, collect(c.titolo) AS corsi, sum(c.cfu) AS cfu_totali
    ORDER BY cfu_totali DESC
""")

### WITH: comporre query a più passi

`WITH` funziona come una "pipeline": passa i risultati intermedi alla parte successiva della query, permettendo di filtrare sui valori aggregati (l'equivalente di `HAVING` in SQL).

In [ ]:
# Studenti con almeno 3 esami superati e media >= 25
run("""
    MATCH (s:Studente)-[r:ISCRITTO_A]->(:Corso)
    WHERE r.voto IS NOT NULL
    WITH s, count(r) AS esami, avg(r.voto) AS media
    WHERE esami >= 3 AND media >= 25
    RETURN s.nome AS studente, esami, round(media, 2) AS media
    ORDER BY media DESC
""")

### OPTIONAL MATCH

Funziona come una `LEFT JOIN`: se il pattern non viene trovato, restituisce `null` invece di scartare la riga.

In [ ]:
run("""
    MATCH (c:Corso)
    OPTIONAL MATCH (s:Studente)-[r:ISCRITTO_A]->(c)
    WHERE r.voto = 30
    RETURN c.titolo AS corso, collect(s.nome) AS studenti_con_30
    ORDER BY size(studenti_con_30) DESC
""")

### Sottoquery: `COUNT { }` ed `EXISTS { }`

In [ ]:
run("""
    MATCH (s:Studente)
    WHERE NOT EXISTS { (s)-[:ISCRITTO_A]->(:Corso)<-[:INSEGNA]-(:Docente {nome: 'Laura Bianchi'}) }
    RETURN s.nome AS studente,
           COUNT { (s)-[:AMICO_DI]-() } AS amici
    ORDER BY amici DESC
    LIMIT 5
""")

## 8. Navigare il grafo

Qui i database a grafo danno il meglio: le query che in SQL richiederebbero molti JOIN (o query ricorsive) in Cypher sono semplici pattern.

### Direzione delle relazioni

Le relazioni hanno sempre una direzione, ma in una query si può ignorarla omettendo la freccia: `-[:AMICO_DI]-`. Per l'amicizia, che è simmetrica, è proprio quanto serve.

In [ ]:
run("""
    MATCH (s:Studente {nome: 'Giorgio Romano'})-[:AMICO_DI]-(amico)
    RETURN amico.nome AS amici_di_giorgio
""")

### Raccomandazioni: "amici di amici"

Si suggeriscono nuove amicizie: persone a distanza 2 che non sono già amiche, ordinate per numero di amici in comune.

In [ ]:
run("""
    MATCH (me:Studente {nome: 'Giorgio Romano'})-[:AMICO_DI]-(comune)-[:AMICO_DI]-(suggerito)
    WHERE suggerito <> me AND NOT (me)-[:AMICO_DI]-(suggerito)
    RETURN suggerito.nome AS suggerito, collect(comune.nome) AS amici_in_comune
    ORDER BY size(amici_in_comune) DESC
""")

In [ ]:
# Corsi consigliati: quelli seguiti dagli amici ma non dallo studente
run("""
    MATCH (me:Studente {nome: 'Giorgio Romano'})-[:AMICO_DI]-(amico)-[:ISCRITTO_A]->(c:Corso)
    WHERE NOT (me)-[:ISCRITTO_A]->(c)
    RETURN c.titolo AS corso_consigliato, count(DISTINCT amico) AS amici_iscritti
    ORDER BY amici_iscritti DESC
""")

### Cammini di lunghezza variabile

`-[:AMICO_DI]-{1,3}` significa "da 1 a 3 salti" (*quantified path pattern*). Con la forma `+` si intende "uno o più".

In [ ]:
run("""
    MATCH (me:Studente {nome: 'Giorgio Romano'})-[:AMICO_DI]-{1,3}(altro:Studente)
    WHERE altro <> me
    RETURN count(DISTINCT altro) AS raggiungibili_in_3_salti
""")

### Cammini minimi

`SHORTEST k` restituisce i `k` cammini più corti tra due nodi; `ALL SHORTEST` li restituisce tutti (a parità di lunghezza). Assegnando il cammino a una variabile (`p = ...`) si possono usare `nodes(p)`, `relationships(p)` e `length(p)`.

In [ ]:
run("""
    MATCH p = SHORTEST 1 (a:Studente {nome: 'Giorgio Romano'})-[:AMICO_DI]-+(b:Studente {nome: 'Viola Colombo'})
    RETURN length(p) AS salti, [n IN nodes(p) | n.nome] AS percorso
""")

In [ ]:
# Sei gradi di separazione? La distanza massima tra due studenti (il "diametro" della rete)
run("""
    MATCH p = ALL SHORTEST (a:Studente)-[:AMICO_DI]-+(b:Studente)
    WHERE a.matricola < b.matricola
    RETURN max(length(p)) AS diametro, round(avg(length(p)), 2) AS distanza_media
""")

Un pattern può attraversare relazioni di tipo diverso. Che legame c'è tra due docenti di dipartimenti diversi?

In [ ]:
run("""
    MATCH p = SHORTEST 1 (a:Docente {nome: 'Laura Bianchi'})-[:INSEGNA|ISCRITTO_A|AMICO_DI]-+(b:Docente {nome: 'Sara Conti'})
    RETURN [n IN nodes(p) | coalesce(n.titolo, n.nome)] AS percorso,
           [r IN relationships(p) | type(r)] AS relazioni
""")

## 9. Visualizzare il grafo

### neo4j-viz

La libreria ufficiale [`neo4j-viz`](https://neo4j.com/docs/nvl-python/) disegna un grafo interattivo direttamente nel notebook: si può zoomare, trascinare i nodi e passare con il mouse per vedere le proprietà.

Per visualizzare un grafo, la query deve restituire **nodi e relazioni** (non solo proprietà). Il driver li raccoglie con `result_transformer_=neo4j.Result.graph`.

In [ ]:
from neo4j_viz.neo4j import from_neo4j

grafo = driver.execute_query("""
    MATCH (s:Studente)-[a:AFFERISCE_A]->(d:Dipartimento)
    OPTIONAL MATCH (s)-[r:AMICO_DI]->(t:Studente)
    RETURN s, a, d, r, t
""", result_transformer_=neo4j.Result.graph)

VG = from_neo4j(grafo)          # i nodi sono colorati in base all'etichetta
for nodo in VG.nodes:           # come didascalia si mostra il nome invece dell'etichetta
    nodo.caption = nodo.properties.get("nome")
VG.render(initial_zoom=0.8)

### NetworkX

I dati possono anche essere esportati verso altre librerie Python. Qui viene costruito un grafo [NetworkX](https://networkx.org/) delle amicizie, disegnato con matplotlib e colorato per dipartimento.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

amicizie = run("""
    MATCH (a:Studente)-[:AMICO_DI]->(b:Studente)
    RETURN a.nome AS a, b.nome AS b
""")
dipartimenti = run("""
    MATCH (s:Studente)-[:AFFERISCE_A]->(d:Dipartimento)
    RETURN s.nome AS nome, d.codice AS dip
""").set_index("nome")["dip"]

G = nx.from_pandas_edgelist(amicizie, "a", "b")
G.add_nodes_from(dipartimenti.index)     # include anche eventuali studenti senza amici
colori = {"INF": "tab:blue", "MAT": "tab:orange", "ECO": "tab:green"}

plt.figure(figsize=(11, 8))
nx.draw_networkx(G, pos=nx.spring_layout(G, seed=7), node_color=[colori[dipartimenti[n]] for n in G.nodes],
                 node_size=500, font_size=8, edge_color="lightgray")
plt.title("Rete di amicizie (blu = INF, arancione = MAT, verde = ECO)")
plt.axis("off")
plt.show()

## 10. Neo4j Browser

Oltre ai driver, Neo4j offre un'interfaccia web per scrivere query e vedere i risultati come grafo.

1. aprire <http://localhost:7474> nel browser;
2. connettersi con URL `neo4j://localhost:7687`, utente `neo4j` e la password del laboratorio (predefinita: `laboratorio`);
3. eseguire, ad esempio, queste query:

```cypher
// L'intero grafo degli studenti di Informatica
MATCH (s:Studente)-[:AFFERISCE_A]->(:Dipartimento {codice: 'INF'})
OPTIONAL MATCH (s)-[r:AMICO_DI]-(t)
RETURN s, r, t

// Lo schema del database
CALL db.schema.visualization()

// Il percorso tra due docenti
MATCH p = SHORTEST 1 (a:Docente {nome: 'Laura Bianchi'})-[*]-(b:Docente {nome: 'Sara Conti'})
RETURN p
```

Comandi utili nel Browser: `:help`, `:schema`, `:sysinfo`, `:clear`.

---
Al termine, la connessione va chiusa:

In [ ]:
driver.close()